In [3]:
%pip install -q --upgrade --force-reinstall langchain-groq langchain-core

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-genai 2.12.1 requires tenacity<9.2.0,>=8.2.3, but you have tenacity 9.2.1 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 17.4 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 17.4 which is incompatible.


In [3]:
from langchain_groq import ChatGroq
from pathlib import Path
import os
from google.colab import userdata
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [4]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [5]:
llm = ChatGroq(
        model = "openai/gpt-oss-120b",
        model_kwargs= {"seed" : 350 },
        temperature= 0,
        max_tokens= 250
)

Runnable pipeline :

In [6]:
template = ChatPromptTemplate.from_messages([
    ('system' , 'you are a lively chatbot that answers in a lively way'),
    ('human' , "it's cold weather tonight isn't it ?")
])

In [7]:
chain = template | llm | StrOutputParser()

response = chain.invoke({})

print(response)

Absolutely! ❄️ The night’s got that crisp, “grab‑a‑blanket” vibe going on. Perfect excuse for a cozy sweater, a steaming mug of cocoa, or even a quick dance‑party in your living room to warm those toes up! How are you staying toasty? 🌙✨


Batching

In [8]:
batch_template = ChatPromptTemplate.from_messages([
    ('system' , 'you will be given two words , use them in one sentence'),
    ('human' , "{word_1} , {word_2}")
])

In [9]:
chain = batch_template | llm | StrOutputParser()

In [10]:
%time
print(chain.invoke({'word_1' : 'hello' , 'word_2' : 'bright'}))
%time
print(chain.invoke({'word_1' : 'shuffle' , 'word_2' : 'proud'}))

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 5.96 µs
Hello, bright sunrise, you’ve already lifted my spirits for the day.
CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 4.77 µs
I felt proud as I watched the dancers shuffle across the stage, each step perfectly timed to the rhythm.


In [11]:
%time
chain.batch(inputs= [
    {'word_1' : 'laptop' , 'word_2' : 'cat'},
    {'word_1' : 'sand' , 'word_2' : 'phone'}
])

CPU times: user 2 µs, sys: 1 µs, total: 3 µs
Wall time: 5.01 µs


['The cat curled up on the warm laptop, purring as the screen glowed softly.',
 'The phone slipped from his hand and landed in the sand, sending a tiny cloud of grains into the air.']

Stream:

In [12]:
stream_template = ChatPromptTemplate.from_messages([
    ('system' , 'you will be given a topic, generate some point on it'),
    ('human' , '{topic}')
])

In [13]:
chain = stream_template | llm | StrOutputParser()
response = chain.stream({'gta 6'})

#next(response)

for i in response:
   print(i, end= '')



**Grand Theft Auto VI – Key Points & What to Expect**

| Category | Details & Highlights |
|----------|----------------------|
| **Current Status** | • Rockstar Games has officially confirmed that *Grand Theft Auto VI* is in development (announced in early 2022). <br>• No exact release date has been announced yet, but industry analysts expect a launch sometime in 2025‑2026. |
| **Potential Setting** | • Rumors point to a **Vice City‑inspired** location (a modern, neon‑lit version of Miami). <br>• Some leaks suggest a **dual‑city** structure, possibly combining a coastal metropolis with an inland, more rural region (e.g., a “Vice City + Liberty City” hybrid). |
| **Story & Protagonists** | • Early leaks hint at **multiple playable protagonists**, similar to *G